# General Code

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

In [17]:
df = pd.read_csv('raw/AOIBaseTFG.tsv', sep='\t')

df['Peak_velocity_of_entry_saccade'] = df['Peak_velocity_of_entry_saccade'].str.replace(',', '.').astype(float)
df['Peak_velocity_of_exit_saccade'] = df['Peak_velocity_of_exit_saccade'].str.replace(',', '.').astype(float)

claro = df[df['Timeline'] == 'Timeline1-Mode1']
oscuro = df[df['Timeline'] == 'Timeline2-Mode2']


## Basic functions

In [18]:
def obtainBasicData(array):
    mean = np.mean(array)
    
    std = np.std(array)
    max = np.max(array)
    min = np.min(array)
    return mean, std, max, min

def printBasicData(array, label):
    mean, std, max, min = obtainBasicData(array)
    print(f"\n{label} - Tiempo hasta la primera fijación")
    print(f"Mean: {mean:.2f},  Std: {std:.2f}, Max: {max:.2f}, Min: {min:.2f}")

# Test function i made to understand how to do 
def ttFirstFixation(aoi, toiClaro = 'Entire Recording', toiOscuro = 'Entire Recording'):
    ttf_claro = claro[claro['AOI'] == aoi]
    ttf_claro = ttf_claro[ttf_claro['TOI'] == toiClaro]['Time_to_first_fixation']
    
    ttf_oscuro = oscuro[oscuro['AOI'] == aoi]
    ttf_oscuro = ttf_oscuro[ttf_oscuro['TOI'] == toiOscuro]['Time_to_first_fixation']

    printBasicData(ttf_claro, f"{aoi} Claro")
    printBasicData(ttf_oscuro, f"{aoi} Oscuro")
    print("\n-------------------------")
    print("Apply test t-student for independent samples:")
    tStat, p_value = stats.ttest_ind(ttf_claro, ttf_oscuro, equal_var=False)
    print(f"T-statistic: {tStat:.4f}, P-value: {p_value:.4f}")
    if(p_value < 0.05):
        print("The difference is statistically significant.")
    return ttf_claro, ttf_oscuro

def timeToFirstFixationPlot(ttf_claro, ttf_oscuro, aoi):
    labels = ['Claro', 'Oscuro']
    data = [ttf_claro, ttf_oscuro]

    plt.figure(figsize=(8, 6))
    plt.boxplot(data, labels=labels)
    plt.title(f'Time to First Fixation for AOI: {aoi}')
    plt.ylabel('Time to First Fixation (ms)')
    plt.grid(axis='y')
    plt.show()
    
def saveToCSV(dataDict, filename):
    with open("output/" + filename, 'w') as f:
        f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
        for aoi, data in dataDict.items():
            f.write(f"{aoi},{data.claroMean:.3f},{data.claroStd:.3f},{data.claroMax:.3f},{data.claroMin:.3f},{data.claroCount},{data.oscuroMean:.3f},{data.oscuroStd:.3f},{data.oscuroMax:.3f},{data.oscuroMin:.3f},{data.oscuroCount},{data.tStat:.5f},{data.pValue:.5f},Yes\n")


## Example of experimental function

In [19]:
ttf_titulo_claro, ttf_titulo_oscuro = ttFirstFixation('Titulo')


Titulo Claro - Tiempo hasta la primera fijación
Mean: 112888.90,  Std: 20551.21, Max: 149679.00, Min: 85900.00

Titulo Oscuro - Tiempo hasta la primera fijación
Mean: 114903.50,  Std: 23804.81, Max: 180827.00, Min: 87543.00

-------------------------
Apply test t-student for independent samples:
T-statistic: nan, P-value: nan


## Bulk function

In [20]:
class TimeToFirstFixationObject:
    claroMean: float
    claroStd: float
    oscuroMean: float
    oscuroStd: float
    aoi: str
    pValue: float
    tStat: float
    claroMax: float
    claroMin: float
    oscuroMax: float
    oscuroMin: float
    oscuroCount: int
    claroCount: int
    

def timeToFirstFixationBulk(column, toiClaro = 'Entire Recording', toiOscuro = 'Entire Recording',csv= True, csvName = "bulk.csv"):
    claro_bulk = df[df['Timeline'] == 'Timeline1-Mode1']
    oscuro_bulk = df[df['Timeline'] == 'Timeline2-Mode2']
    
    
    
    claro_bulk = claro_bulk[claro_bulk[column] > 0] # It can make noise in the data
    oscuro_bulk = oscuro_bulk[oscuro_bulk[column] > 0] 

    aois = df['AOI'].unique()
    significativeResults = {}
    noSignificativeResults = {}
    
    for aoi in aois:
        aoiObject = TimeToFirstFixationObject()
        ttf_claro = claro_bulk[claro_bulk['AOI'] == aoi]
        ttf_claro = ttf_claro[ttf_claro['TOI'] == toiClaro][column]
        ttf_claro = ttf_claro.dropna()
        
        ttf_oscuro = oscuro_bulk[oscuro_bulk['AOI'] == aoi]
        ttf_oscuro = ttf_oscuro[ttf_oscuro['TOI'] == toiOscuro][column]
        ttf_oscuro = ttf_oscuro.dropna()
        
        # Skip if either dataset is empty
        if len(ttf_claro) == 0 or len(ttf_oscuro) == 0:
            continue
            
        aoiObject.claroMean, aoiObject.claroStd, aoiObject.claroMax, aoiObject.claroMin = obtainBasicData(ttf_claro)
        aoiObject.oscuroMean, aoiObject.oscuroStd, aoiObject.oscuroMax, aoiObject.oscuroMin = obtainBasicData(ttf_oscuro)
        aoiObject.aoi = aoi
        
        tStat, p_value = stats.ttest_ind(ttf_claro, ttf_oscuro, equal_var=False)
        aoiObject.pValue = p_value
        aoiObject.tStat = tStat
        aoiObject.claroCount = len(ttf_claro)
        aoiObject.oscuroCount = len(ttf_oscuro)
        if(p_value < 0.05):
            significativeResults[aoi] = aoiObject
        else:
            noSignificativeResults[aoi] = aoiObject
        
    if csv:
        saveToCSV(significativeResults, "sig_" + csvName)
        saveToCSV(noSignificativeResults, "no_sig_" + csvName)

    return significativeResults, noSignificativeResults

## Bulk function results
### Sample data

In [21]:
toiClaro = ['PatataClaro', 'MochilaClaro','Text']
toiOscuro = ['PatataOscuro', 'MochilaOscuro','Text1']
columnNames = ['Total_duration_of_fixations', 'Number_of_fixations', 'Time_to_first_fixation', 'Number_of_saccades_in_AOI', 'Time_to_entry_saccade', 'Time_to_exit_saccade', 'Peak_velocity_of_entry_saccade', 'Peak_velocity_of_exit_saccade']
csvNames = ['tdof.csv', 'nof.csv', 'ttff.csv', 'nosia.csv', 'ttents.csv', 'ttexts.csv', 'pvents.csv', 'pvexts.csv']

In [22]:
for i in range(len(columnNames)):
    print(f"\n\nAnalyzing column: {columnNames[i]}\n")
    significantResults = {}
    nonSignificantResults = {}
    for j in range(len(toiClaro)):
        significant, nonSignificant = timeToFirstFixationBulk(columnNames[i], toiClaro[j], toiOscuro[j], csvName= csvNames[i])
        significantResults.update(significant)
        nonSignificantResults.update(nonSignificant)
    print(f"Total significant results for column {columnNames[i]}: {len(significantResults)}")
    print(f"Total non significant results for column {columnNames[i]}: {len(nonSignificantResults)}")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")



Analyzing column: Total_duration_of_fixations



c:\Users\garab\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Total significant results for column Total_duration_of_fixations: 21
Total non significant results for column Total_duration_of_fixations: 487
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Analyzing column: Number_of_fixations

Total significant results for column Number_of_fixations: 8
Total non significant results for column Number_of_fixations: 497
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Analyzing column: Time_to_first_fixation

Total significant results for column Time_to_first_fixation: 33
Total non significant results for column Time_to_first_fixation: 480
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Analyzing column: Number_of_saccades_in_AOI

Total significant results for column Number_of_saccades_in_AOI: 0
Total non significant results for column Number_of_saccades_in_AOI: 68
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Analyzing column: Time_to_entry_saccade

Total significant results for column Time_to_entry